# 3, 4, 5, 6 ერთ ფაილში

In [ ]:
# 3, 4, 5 ერთ ფაილში

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt


# ----------------------------------------------------------
# SYMBOLS
# ----------------------------------------------------------

t = sp.Symbol('t', real=True)
eps = sp.Symbol('ε', real=True)


# ----------------------------------------------------------
# CREATE HAMILTONIAN
# ----------------------------------------------------------

def build_hamiltonian(n):
    """
    Creates the 2n x 2n Hamiltonian.

    Ordering:
    n1, nbar1, n2, nbar2, ..., nn, nbarn
    """

    H = sp.zeros(2*n, 2*n)

    for i in range(n):
        for j in range(n):

            if i != j:

                # n_i -> nbar_j
                H[2*i, 2*j + 1] = eps

                # nbar_i -> n_j
                H[2*i + 1, 2*j] = eps

    return H

t_vals = np.logspace(
    -2,
    4,
    2000
)
# ----------------------------------------------------------
# MAIN PROGRAM
# ----------------------------------------------------------

n = int(input("How many sectors do you want? "))


# ----------------------------------------------------------
# tau_n AND epsilon
# ----------------------------------------------------------

tau_n = 400 * np.sqrt(n - 1)

eps_val = 1 / tau_n

print("\nNumber of sectors =", n)
print("tau_n =", tau_n)
print("epsilon =", eps_val)


# ----------------------------------------------------------
# HAMILTONIAN
# ----------------------------------------------------------

H = build_hamiltonian(n)

print("\nHamiltonian:")
sp.pprint(H)


# ----------------------------------------------------------
# EVOLUTION MATRIX
#
#       S = exp(-i H t)
#
# EXACTLY AS IN YOUR ORIGINAL CODE
# ----------------------------------------------------------

print("\nCalculating S = exp(-i H t)...")

S = sp.exp(-sp.I * H * t)

print("Done.")


# ----------------------------------------------------------
# PROBABILITY MATRIX
#
#       P_ij = S_ij S_ij*
# ----------------------------------------------------------

dim = 2*n

P = sp.zeros(dim, dim)

print("\nCalculating probabilities...")

for i in range(dim):

    for j in range(dim):

        P[i, j] = sp.simplify(
            sp.expand_complex(
                S[i, j] * S[i, j].conjugate()
            )
        )

print("Done.")


# ----------------------------------------------------------
# PROBABILITIES FROM INITIAL n STATE
#
# Initial state = index 0
# ----------------------------------------------------------

P_nn = sp.simplify(P[0, 0])

P_nbar_n = sp.simplify(P[0, 1])


# everything outside first sector
P_nx = 0

for j in range(2, dim):

    P_nx += P[0, j]

P_nx = sp.simplify(P_nx)


# ----------------------------------------------------------
# CHECK TOTAL PROBABILITY
# ----------------------------------------------------------

P_total = 0

for j in range(dim):

    P_total += P[0, j]

P_total = sp.simplify(P_total)


print("\nP_nn:")
sp.pprint(P_nn)

print("\nP_nbar_n:")
sp.pprint(P_nbar_n)

print("\nSum P_nx:")
sp.pprint(P_nx)

print("\nTotal probability:")
sp.pprint(P_total)


# ----------------------------------------------------------
# NUMERICAL PARAMETERS
# ----------------------------------------------------------

tau_nn = 0.9 * 10**8

tau_decay = 880


# ----------------------------------------------------------
# P_nbn
# Same definition as your original code
# ----------------------------------------------------------

P_nbn = sp.sin(
    t * tau_n / tau_nn
)**2


# ----------------------------------------------------------
# DECAY FACTOR
# ----------------------------------------------------------

decay = sp.exp(-t_vals / tau_decay)


# ----------------------------------------------------------
# SUBSTITUTE epsilon
# ----------------------------------------------------------

P_nn_num = P_nn.subs(eps, eps_val)

P_nbar_n_num = P_nbar_n.subs(eps, eps_val)

P_nx_num = P_nx.subs(eps, eps_val)


# include decay
P_nn_num = P_nn_num * decay

P_nbar_n_num = P_nbar_n_num * decay

P_nx_num = P_nx_num * decay

P_nbn_num = P_nbn * decay


# ----------------------------------------------------------
# CONVERT SYMPY EXPRESSIONS TO NUMPY FUNCTIONS
# ----------------------------------------------------------

f_nn = sp.lambdify(
    t,
    P_nn_num,
    "numpy"
)

f_nbar_n = sp.lambdify(
    t,
    P_nbar_n_num,
    "numpy"
)

f_nx = sp.lambdify(
    t,
    P_nx_num,
    "numpy"
)

f_nbn = sp.lambdify(
    t,
    P_nbn_num,
    "numpy"
)


# ----------------------------------------------------------
# TIME VALUES
# ----------------------------------------------------------

t_vals = np.logspace(
    -2,
    4,
    2000
)


# ----------------------------------------------------------
# CALCULATE CURVES
# ----------------------------------------------------------

params = {
    eps: 1,
    #m: 1.0
}
P_dict_n = {
    r"$P_{nn}$": P[0,0],
    r"$P_{n\bar n}$": P[0,1],
    r"$P_{nbn}$": P_nbn,
    #r"$\sum{P_{n\bar n_i}}$": s_0ib,
    r"$\sum{P_{nx}}$": P_nx,
}

P_dict_p = {
    r"$P_{nn}$":P[0,0],
    r"$P_{n\bar n}$": P[0,1],
    r"$\sum{P_{nx}}$": P_nx,
    r"$P_{nbn}$": P_nbn,
    #r"$\sum{P_{n\bar n_i}}$": s_0ib,
}
styles = [
    {"color": "black", "linestyle": "-"},
    {"color": "red", "linestyle": "-"},
    {"color": "blue", "linestyle": "--"},
    #{"color": "orange", "linestyle": "-"},
    {"color": "green", "linestyle": "-"},
]
t_vals = np.logspace(-2, 4, 2000)
P_funcs_3 = {label: sp.lambdify(t, (expr*sp.exp(-eps_val*t_vals*tau_n/tau_decay)).subs(params), "numpy")
    for label, expr in P_dict_n.items()
}
P_funcs_p = {label: sp.lambdify(t, (expr*sp.exp(-eps_val*t_vals*tau_n/tau_decay)).subs(params), "numpy")
    for label, expr in P_dict_p.items()
}
sample_expr = next(iter(P_dict_n.values()))
extra_symbols = list(sample_expr.free_symbols - {t})
#print("extra symbols in expressions:", extra_symbols)

f = P_funcs_3[r"$P_{n\bar n}$"]
f_x = P_funcs_3[r"$\sum{P_{nx}}$"]

# Evaluate it on the plotted grid
y = f(eps_val * t_vals)

# Find the maximum
idx = np.argmax(y)

t_max = t_vals[idx]
P_max = y[idx]

print(n," sectors")

print(f"Maximum = {P_max:.6e}")
print(f"Occurs at t = {t_max:.6e} s")


times = np.array([0.1, 1, 10, 100, 1000])

# Header
print(f"{'t (s)':>8} {'P_nn':>12} {'P_nbar n':>12} {'Sum P_nx':>12} {'P_nbn':>12}")
print("-"*62)

# Compute all curves only once
curves = {}
for label, f in P_funcs_p.items():
    curves[label] = f(eps_val * t_vals)

# Print interpolated values
for tq in times:
    values = []
    for label in P_dict_p.keys():
        values.append(np.interp(tq, t_vals, curves[label]))

    print(f"{tq:8.1f} "
          f"{values[0]:12.4e} "
          f"{values[1]:12.4e} "
          f"{values[2]:12.4e} "
          f"{values[3]:12.4e}")
# Build a substitution map for every non-t symbol
subs_map = {}
for sym in extra_symbols:
    if sym.name in ("eps", "ε"):
        subs_map[sym] = eps_val
    elif sym.name == "m":
        subs_map[sym] = 1.0

# Check again after substitution
#for label, expr in P_dict_3.items():
#    print(label, (expr.subs(subs_map)).free_symbols)
plt.figure(figsize=(8,5))

for (label, f), style in zip(P_funcs_3.items(), styles):
    plt.loglog(t_vals, f(eps_val*t_vals), label=label, linewidth=2, **style)

plt.loglog(
    t_vals,
    np.exp(-eps_val*t_vals*tau_n/tau_decay),
    color="black",
    linestyle="--",
    linewidth=2,
    label=r"$e^{-t/\tau}$"
)
plt.xlabel(r"$t~[s]$", fontsize=16)
plt.ylabel(r"$P(t)$", fontsize=16)

plt.xlim(1e-2, 1e4)
plt.ylim(1e-20, 2)
plt.axhline(y=6.25*10**(-8), color='green', linestyle='--', linewidth=1.5)
plt.axhline(y=10**(-18), color='orange', linestyle='--', linewidth=1.5)
plt.text(
    1e0, 1e-1,                      # position (t, P)
    r"$P_{nn}$",
    fontsize=14
)
plt.text(
    1e1, 1e-9,
    # position (t, P)
    r"$P_{n\bar n}$",
    fontsize=14,
)
plt.text(
    1e1, 1e-6,                      # position (t, P)
    r"$P_{nx}$",
    fontsize=14,
)
plt.text(
    1e3, 1e-1,                      # position (t, P)
    r"$e^{-t/\tau}$",
    fontsize=14,
)
plt.legend(
    ncol=5,
    fontsize=12,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.15),
    frameon=False
)
plt.grid(True, which="both")
plt.title("n  sectors", y=1.13)
plt.tight_layout()
plt.show()

How many sectors do you want? 15

Number of sectors = 15
tau_n = 1496.6629547095765
epsilon = 0.000668153104781061

Hamiltonian:
⎡0  0  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  ↪
⎢                                                                              ↪
⎢0  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ↪
⎢                                                                              ↪
⎢0  ε  0  0  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  ↪
⎢                                                                              ↪
⎢ε  0  0  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ↪
⎢                                                                              ↪
⎢0  ε  0  ε  0  0  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  ↪
⎢                                                                              ↪
⎢ε  0  ε  0  0  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  ε  0  